In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
"""
영양 중재 효과 시뮬레이터 (Rule-Based Model)
- MIMIC-IV ML 모델 (Albumin)과 결합 가능
- 문헌 기반 가이드라인으로 다른 영양소 효과 예측
"""

import warnings
from typing import Dict, Any, Optional, List
from dataclasses import dataclass, field


@dataclass
class PatientProfile:
    """환자 프로필"""
    age: int
    sex: str  # 'M' or 'F'

    # Lab values
    hemoglobin: Optional[float] = None  # g/dL
    ferritin: Optional[float] = None  # ng/mL
    tsat: Optional[float] = None  # %
    albumin: Optional[float] = None  # g/dL
    vitamin_d: Optional[float] = None  # 25(OH)D ng/mL
    calcium: Optional[float] = None  # mg/dL
    crp: Optional[float] = None  # mg/L

    # Clinical conditions
    ckd_stage: int = 0  # 0=none, 1-5=CKD stages
    smoker: bool = False
    immune_compromised: bool = False
    chronic_inflammation: bool = False
    kidney_stone_history: bool = False
    hemochromatosis: bool = False
    hypercalcemia: bool = False

    # Additional factors
    fracture_risk_high: bool = False


@dataclass
class NutritionIntervention:
    """영양 중재 계획"""
    # 영양소 용량 (일일)
    iron_mg: Optional[float] = None  # elemental iron
    vitamin_d_iu: Optional[float] = None
    calcium_mg: Optional[float] = None
    omega3_epa_dha_g: Optional[float] = None
    vitamin_c_mg: Optional[float] = None
    protein_g: Optional[float] = None  # for albumin (ML model 사용)

    # 기간
    duration_weeks: int = 4


@dataclass
class SimulationResult:
    """시뮬레이션 결과"""
    parameter: str
    current_value: Optional[float]
    expected_value: Optional[float]
    expected_change: Optional[float]
    interpretation: str
    warnings: List[str] = field(default_factory=list)
    contraindications: List[str] = field(default_factory=list)
    monitoring_recommendations: List[str] = field(default_factory=list)


class NutritionSimulator:
    """영양 중재 효과 시뮬레이터"""

    def __init__(self, albumin_model=None):
        """
        Args:
            albumin_model: MIMIC-IV 기반 ML 모델 (optional)
        """
        self.albumin_model = albumin_model
        self._load_guidelines()

    def _load_guidelines(self):
        """문헌 기반 가이드라인 로드"""
        self.guidelines = {
            'iron': {
                'absolute_deficiency': {
                    'hgb_threshold': 12.0,  # g/dL
                    'ferritin_threshold': 30.0,  # ng/mL
                    'tsat_threshold': 20.0  # %
                },
                'avoid_supplementation': {
                    'ferritin_high': 300.0,  # ng/mL
                    'tsat_high': 45.0  # %
                },
                'treatment_dose': (100, 200),  # mg/day range
                'baseline_hgb_increase': 1.0,  # g/dL per 3-4 weeks
                'ckd_absorption_factor': 0.7,
                'inflammation_factor': 0.7,
            },
            'vitamin_d': {
                'sufficient_threshold': 20.0,  # ng/mL
                'optimal_range': (20, 30),  # ng/mL
                'deficiency_moderate': 10.0,  # ng/mL
                'maintenance_dose': (800, 1000),  # IU/day
                'repletion_dose': (1000, 2000),  # IU/day
                'upper_limit': 4000,  # IU/day
                'increase_per_100iu': 1.5,  # ng/mL per 100 IU/day over 3-6 months
            },
            'calcium': {
                'target_total': (1000, 1200),  # mg/day
                'supplement_dose': (500, 600),  # mg per dose
                'fracture_risk_reduction': 0.15,  # 15% relative risk reduction
                'timeframe_months': 12,
            },
            'omega3': {
                'general_elderly': (0.25, 0.5),  # g EPA+DHA/day
                'high_risk_cvd': (0.8, 1.2),  # g EPA+DHA/day
                'mace_reduction': 0.08,  # 8% reduction estimate
            },
            'vitamin_c': {
                'rni': 100,  # mg/day (한국인 권장섭취량)
                'optimal_range': (200, 500),  # mg/day for clinical benefit
                'upper_limit': 2000,  # mg/day
                'kidney_stone_risk_threshold_male': 1000,  # mg/day
                'plasma_saturation_dose': 200,  # mg/day
            }
        }

    def simulate(self, patient: PatientProfile, intervention: NutritionIntervention) -> Dict[str, SimulationResult]:
        """
        영양 중재 효과 시뮬레이션

        Args:
            patient: 환자 프로필
            intervention: 영양 중재 계획

        Returns:
            Dict[영양소, SimulationResult]
        """
        results = {}

        # 1. Iron (Hemoglobin)
        if intervention.iron_mg is not None:
            results['hemoglobin'] = self._simulate_iron(patient, intervention)

        # 2. Vitamin D
        if intervention.vitamin_d_iu is not None:
            results['vitamin_d'] = self._simulate_vitamin_d(patient, intervention)

        # 3. Calcium (Fracture Risk)
        if intervention.calcium_mg is not None:
            results['fracture_risk'] = self._simulate_calcium(patient, intervention)

        # 4. Omega-3 (Cardiovascular Risk)
        if intervention.omega3_epa_dha_g is not None:
            results['cvd_risk'] = self._simulate_omega3(patient, intervention)

        # 5. Vitamin C
        if intervention.vitamin_c_mg is not None:
            results['vitamin_c'] = self._simulate_vitamin_c(patient, intervention)

        # 6. Protein → Albumin (ML model 사용)
        if intervention.protein_g is not None and self.albumin_model is not None:
            results['albumin'] = self._simulate_albumin(patient, intervention)

        return results

    def _simulate_iron(self, patient: PatientProfile, intervention: NutritionIntervention) -> SimulationResult:
        """철분 보충 → Hemoglobin 변화 예측"""
        gl = self.guidelines['iron']
        warnings_list = []
        contraindications = []
        monitoring = ['Hgb every 2-4 weeks', 'Ferritin/TSAT every 2-3 months']

        # 금기사항 체크
        if patient.hemochromatosis:
            contraindications.append('혈색소침착증 - 철분 보충 금기')

        if patient.ferritin is not None and patient.ferritin > gl['avoid_supplementation']['ferritin_high']:
            contraindications.append(f'고페리틴혈증 (Ferritin {patient.ferritin:.1f} ng/mL > 300) - 철분 축적 위험')
            expected_change = 0.0
            interpretation = '철분 보충 권장하지 않음 (다른 빈혈 원인 평가 필요)'

        elif patient.tsat is not None and patient.tsat > gl['avoid_supplementation']['tsat_high']:
            contraindications.append(f'TSAT 과다 ({patient.tsat:.1f}% > 45%) - 철분 과부하 위험')
            expected_change = 0.0
            interpretation = '철분 보충 권장하지 않음'

        else:
            # 기본 반응
            baseline_increase = gl['baseline_hgb_increase']  # 1.0 g/dL per 3-4 weeks

            # 환자 특성에 따른 보정
            ckd_factor = gl['ckd_absorption_factor'] if patient.ckd_stage >= 3 else 1.0
            inflammation_factor = gl['inflammation_factor'] if (patient.crp and patient.crp > 5.0) or patient.chronic_inflammation else 1.0

            # 용량 보정 (100-200mg 범위)
            dose_factor = min(intervention.iron_mg / 150, 1.2)  # 150mg 기준, 최대 1.2배

            # 기간 보정
            weeks_factor = intervention.duration_weeks / 4.0  # 4주 기준

            expected_change = baseline_increase * ckd_factor * inflammation_factor * dose_factor * weeks_factor

            # 경고사항
            if patient.ckd_stage >= 3:
                warnings_list.append(f'CKD stage {patient.ckd_stage} - 철분 흡수율 감소 (예상 효과 70%)')

            if patient.chronic_inflammation or (patient.crp and patient.crp > 5.0):
                warnings_list.append('만성염증 존재 - 기능적 철결핍 가능성 (철분 단독으로 불충분할 수 있음)')

            if patient.age >= 75:
                warnings_list.append('고령 환자 - 혼합형 빈혈 가능성 (B12/엽산, CKD 등 동반 평가 권장)')

            interpretation = f'{intervention.duration_weeks}주 후 Hgb {expected_change:+.1f} g/dL 변화 예상 (문헌 기반 추정)'

        current_hgb = patient.hemoglobin
        expected_hgb = current_hgb + expected_change if current_hgb is not None else None

        return SimulationResult(
            parameter='Hemoglobin',
            current_value=current_hgb,
            expected_value=expected_hgb,
            expected_change=expected_change,
            interpretation=interpretation,
            warnings=warnings_list,
            contraindications=contraindications,
            monitoring_recommendations=monitoring
        )

    def _simulate_vitamin_d(self, patient: PatientProfile, intervention: NutritionIntervention) -> SimulationResult:
        """비타민 D 보충 → 25(OH)D 변화 예측"""
        gl = self.guidelines['vitamin_d']
        warnings_list = []
        contraindications = []
        monitoring = ['25(OH)D and calcium every 3-6 months']

        # 금기사항
        if patient.hypercalcemia:
            contraindications.append('고칼슘혈증 - 비타민 D 보충 금기')
            expected_change = 0.0
            interpretation = '비타민 D 보충 권장하지 않음'

        elif patient.ckd_stage >= 4:
            warnings_list.append('Advanced CKD - 고인산혈증/혈관석회화 위험 (신장내과 협진 필요)')
            expected_change = 0.0
            interpretation = '전문의 상담 후 결정 권장'

        else:
            # 용량에 따른 증가량 계산
            # Rule of thumb: +1-2 ng/mL per 100 IU/day over several months
            increase_per_100iu = gl['increase_per_100iu']
            expected_change = (intervention.vitamin_d_iu / 100) * increase_per_100iu * (intervention.duration_weeks / 12.0)

            # 상한선 체크
            if intervention.vitamin_d_iu > gl['upper_limit']:
                warnings_list.append(f'용량 {intervention.vitamin_d_iu} IU/day가 상한선 {gl["upper_limit"]} IU/day 초과')

            # 현재 수치에 따른 해석
            if patient.vitamin_d is not None:
                if patient.vitamin_d < gl['deficiency_moderate']:
                    interpretation = f'{intervention.duration_weeks}주 후 25(OH)D {expected_change:+.1f} ng/mL 증가 예상 (중등도 결핍 → 보충 필요)'
                elif patient.vitamin_d < gl['sufficient_threshold']:
                    interpretation = f'{intervention.duration_weeks}주 후 25(OH)D {expected_change:+.1f} ng/mL 증가 예상 (경증 결핍 → 보충 권장)'
                else:
                    interpretation = f'{intervention.duration_weeks}주 후 25(OH)D {expected_change:+.1f} ng/mL 증가 예상 (유지 용량)'
            else:
                interpretation = f'{intervention.duration_weeks}주 후 25(OH)D {expected_change:+.1f} ng/mL 증가 예상 (문헌 기반 추정)'

        current_vitd = patient.vitamin_d
        expected_vitd = current_vitd + expected_change if current_vitd is not None else None

        return SimulationResult(
            parameter='25(OH)D',
            current_value=current_vitd,
            expected_value=expected_vitd,
            expected_change=expected_change,
            interpretation=interpretation,
            warnings=warnings_list,
            contraindications=contraindications,
            monitoring_recommendations=monitoring
        )

    def _simulate_calcium(self, patient: PatientProfile, intervention: NutritionIntervention) -> SimulationResult:
        """칼슘 + 비타민 D → 골절 위험 감소"""
        gl = self.guidelines['calcium']
        warnings_list = []
        contraindications = []
        monitoring = ['Serum calcium and renal function periodically']

        # 금기사항
        if patient.kidney_stone_history:
            contraindications.append('신결석 병력 - 칼슘 보충제 주의')

        if patient.hypercalcemia:
            contraindications.append('고칼슘혈증 - 칼슘 보충 금기')

        if patient.ckd_stage >= 4:
            contraindications.append('Advanced CKD - 칼슘 보충 전 신장내과 협진 필요')

        if contraindications:
            risk_reduction_pct = 0.0
            interpretation = '칼슘 보충 권장하지 않음'
        else:
            # 골절 위험 감소는 장기간 (≥1년) 복용 시 효과
            if intervention.duration_weeks >= 12:  # 3개월 이상
                # 문헌: Ca+VitD 병용 시 10-20% 골절 위험 감소
                risk_reduction_pct = gl['fracture_risk_reduction'] * 100  # 15%
                interpretation = f'{intervention.duration_weeks}주 복용 시 골절 위험 약 {risk_reduction_pct:.0f}% 감소 예상 (Ca+VitD 병용 시)'

                if patient.fracture_risk_high:
                    interpretation += ' - 고위험군에서 효과 더 뚜렷함'
            else:
                risk_reduction_pct = gl['fracture_risk_reduction'] * 100 * (intervention.duration_weeks / 12.0)
                warnings_list.append('골절 예방 효과는 1년 이상 장기 복용 시 명확함')
                interpretation = f'{intervention.duration_weeks}주는 단기간 - 골절 위험 감소 효과 제한적'

            # 용량 체크
            if intervention.calcium_mg < gl['target_total'][0]:
                warnings_list.append(f'칼슘 {intervention.calcium_mg}mg/day는 권장량 {gl["target_total"][0]}-{gl["target_total"][1]}mg/day 미만')

            # 부작용 모니터링
            monitoring.append('변비, 신결석 증상 관찰')

        return SimulationResult(
            parameter='Fracture Risk',
            current_value=None,  # 골절 위험은 정량화 어려움
            expected_value=None,
            expected_change=-risk_reduction_pct if risk_reduction_pct > 0 else 0.0,  # 음수 = 위험 감소
            interpretation=interpretation,
            warnings=warnings_list,
            contraindications=contraindications,
            monitoring_recommendations=monitoring
        )

    def _simulate_omega3(self, patient: PatientProfile, intervention: NutritionIntervention) -> SimulationResult:
        """오메가-3 → 심혈관 위험 감소"""
        gl = self.guidelines['omega3']
        warnings_list = []
        contraindications = []
        monitoring = ['심혈관 이벤트 추적']

        # 용량에 따른 효과
        dose = intervention.omega3_epa_dha_g

        if dose >= gl['high_risk_cvd'][0]:  # ≥0.8 g/day
            mace_reduction_pct = gl['mace_reduction'] * 100  # 8%
            interpretation = f'중등도 용량 (≥0.8 g/day) - MACE 약 {mace_reduction_pct:.0f}% 감소 가능 (고위험군 메타분석)'

            if dose > 2.0:
                warnings_list.append('고용량 (>2g/day) - 일부 연구에서 심방세동 위험 소폭 증가 보고')

        elif dose >= gl['general_elderly'][0]:  # 0.25-0.5 g/day
            mace_reduction_pct = gl['mace_reduction'] * 0.5 * 100  # 절반 효과 가정
            interpretation = f'저용량 (0.25-0.5 g/day) - 관상동맥질환 1차 예방 수준'

        else:
            mace_reduction_pct = 0.0
            warnings_list.append('용량 부족 - 심혈관 이벤트 감소 효과 제한적')
            interpretation = '권장 용량 미만'

        # 항응고제 복용 여부 (프로필에 없어서 일반 경고만) v
        warnings_list.append('항응고제 복용 중이면 출혈 위험 주의')

        return SimulationResult(
            parameter='Cardiovascular Risk (MACE)',
            current_value=None,
            expected_value=None,
            expected_change=-mace_reduction_pct if mace_reduction_pct > 0 else 0.0,
            interpretation=interpretation,
            warnings=warnings_list,
            contraindications=contraindications,
            monitoring_recommendations=monitoring
        )

    def _simulate_vitamin_c(self, patient: PatientProfile, intervention: NutritionIntervention) -> SimulationResult:
        """비타민 C 섭취 적정성 평가"""
        gl = self.guidelines['vitamin_c']
        warnings_list = []
        contraindications = []
        monitoring = []

        dose = intervention.vitamin_c_mg

        # 해석 수준
        if dose < gl['rni']:
            level = 'deficient'
            interpretation = f'{dose}mg/day - 권장섭취량 {gl["rni"]}mg/day 미달 (결핍 위험)'

        elif dose < gl['optimal_range'][0]:
            level = 'adequate'
            interpretation = f'{dose}mg/day - 권장섭취량 충족 (결핍 예방 수준)'

        elif dose <= gl['optimal_range'][1]:
            level = 'optimal'
            interpretation = f'{dose}mg/day - 최적 범위 (항산화 효과, 면역 지원)'

        elif dose <= gl['upper_limit']:
            level = 'high_but_safe'
            interpretation = f'{dose}mg/day - 고용량이지만 안전 범위 내 (UL {gl["upper_limit"]}mg 이내)'
            warnings_list.append('1000mg 이상 시 분할 복용 권장 (흡수율 향상)')

        else:
            level = 'excessive'
            interpretation = f'{dose}mg/day - 상한섭취량 초과 (설사, 위장 불편감 위험)'
            warnings_list.append(f'UL {gl["upper_limit"]}mg/day 초과 - 용량 감량 권장')

        # 특수 집단 고려
        if patient.smoker:
            if dose < 150:
                warnings_list.append('흡연자 - 권장 섭취량 +50-100mg (총 150-200mg/day)')

        if patient.immune_compromised or patient.chronic_inflammation:
            if dose < 200:
                warnings_list.append('면역저하/만성염증 - 200-500mg/day 권장')

        # 신결석 위험 (남성만)
        if patient.sex == 'M' and dose >= gl['kidney_stone_risk_threshold_male']:
            if patient.kidney_stone_history:
                contraindications.append('칼슘옥살산 결석 병력 남성 - 보충제 비타민 C (≥1000mg) 피할 것 (신결석 위험 2배)')
            else:
                warnings_list.append('남성 고용량 (≥1000mg) - 신결석 위험 증가 가능 (요중 옥살산 증가)')

        # 철분 과잉 질환
        if patient.hemochromatosis:
            contraindications.append('혈색소침착증 - 비타민 C가 철분 흡수 증가시켜 위험')

        # 철결핍 빈혈이면 유익
        if patient.ferritin is not None and patient.ferritin < 30:
            monitoring.append('철결핍성 빈혈 - 비타민 C와 철분 병용 시 흡수율 67% 증가 (유익)')

        return SimulationResult(
            parameter='Vitamin C Status',
            current_value=None,  # 혈장 비타민 C 농도는 보통 측정 안 함
            expected_value=None,
            expected_change=None,
            interpretation=interpretation,
            warnings=warnings_list,
            contraindications=contraindications,
            monitoring_recommendations=monitoring
        )

    def _simulate_albumin(self, patient: PatientProfile, intervention: NutritionIntervention) -> SimulationResult:
        """단백질 섭취 → Albumin 변화 (ML 모델 사용)"""
        warnings_list = []
        contraindications = []
        monitoring = ['Albumin every 2-4 weeks', 'BUN/Cr 모니터링']

        # Advanced CKD 금기
        if patient.ckd_stage >= 4:
            contraindications.append(f'CKD stage {patient.ckd_stage} - 고단백 섭취 금기 (신기능 악화 위험)')
            expected_change = 0.0
            interpretation = '단백질 제한 필요 (신장내과 협진)'

        else:
            # ML 모델로 예측 (여기서는 placeholder)
            # 실제로는: predicted_albumin = self.albumin_model.predict(patient_features)

            # 임시 규칙 기반 (ML 모델 대체용)
            baseline_increase = 0.3  # g/dL per 4 weeks (문헌)

            # 단백질 용량 (1.0-1.2 g/kg/day 기준)
            # 체중 정보 없으므로 절대량으로 가정
            protein_factor = min(intervention.protein_g / 60, 1.2)  # 60g 기준

            # CKD stage 3 보정
            ckd_factor = 0.8 if patient.ckd_stage == 3 else 1.0

            # 기간 보정
            weeks_factor = intervention.duration_weeks / 4.0

            expected_change = baseline_increase * protein_factor * ckd_factor * weeks_factor

            if patient.albumin is not None and patient.albumin < 3.0:
                interpretation = f'{intervention.duration_weeks}주 후 Albumin {expected_change:+.1f} g/dL 증가 예상 (중등도 결핍 → 보충 필요)'
            elif patient.albumin is not None and patient.albumin < 3.5:
                interpretation = f'{intervention.duration_weeks}주 후 Albumin {expected_change:+.1f} g/dL 증가 예상 (경증 결핍)'
            else:
                interpretation = f'{intervention.duration_weeks}주 후 Albumin {expected_change:+.1f} g/dL 증가 예상'

            if patient.ckd_stage == 3:
                warnings_list.append('CKD stage 3 - 단백질 섭취량 신중히 조절 (과도한 섭취 피할 것)')

        current_albumin = patient.albumin
        expected_albumin = current_albumin + expected_change if current_albumin is not None else None

        return SimulationResult(
            parameter='Albumin',
            current_value=current_albumin,
            expected_value=expected_albumin,
            expected_change=expected_change,
            interpretation=interpretation,
            warnings=warnings_list,
            contraindications=contraindications,
            monitoring_recommendations=monitoring
        )

    def generate_report(self, patient: PatientProfile, intervention: NutritionIntervention,
                       results: Dict[str, SimulationResult]) -> str:
        """시뮬레이션 결과 리포트 생성"""
        report = []
        report.append("=" * 80)
        report.append("영양 중재 효과 예측 시뮬레이션 (문헌 기반)")
        report.append("=" * 80)
        report.append("")

        # 환자 정보
        report.append(f"환자: {patient.age}세, {patient.sex}")
        if patient.ckd_stage > 0:
            report.append(f"  - CKD Stage {patient.ckd_stage}")
        if patient.smoker:
            report.append("  - 흡연자")
        if patient.chronic_inflammation:
            report.append("  - 만성염증 질환")
        report.append("")

        # 중재 계획
        report.append("중재 계획:")
        if intervention.iron_mg:
            report.append(f"  - 철분: {intervention.iron_mg} mg/day")
        if intervention.vitamin_d_iu:
            report.append(f"  - 비타민 D: {intervention.vitamin_d_iu} IU/day")
        if intervention.calcium_mg:
            report.append(f"  - 칼슘: {intervention.calcium_mg} mg/day")
        if intervention.omega3_epa_dha_g:
            report.append(f"  - 오메가-3: {intervention.omega3_epa_dha_g} g EPA+DHA/day")
        if intervention.vitamin_c_mg:
            report.append(f"  - 비타민 C: {intervention.vitamin_c_mg} mg/day")
        if intervention.protein_g:
            report.append(f"  - 단백질: {intervention.protein_g} g/day")
        report.append(f"  - 기간: {intervention.duration_weeks}주")
        report.append("")

        # 결과
        report.append("예측 결과:")
        report.append("-" * 80)

        for param_name, result in results.items():
            report.append(f"\n【{result.parameter}】")

            if result.current_value is not None:
                report.append(f"  현재: {result.current_value:.2f}")

            if result.expected_value is not None:
                report.append(f"  예상: {result.expected_value:.2f} ({result.expected_change:+.2f})")
            elif result.expected_change is not None:
                report.append(f"  예상 변화: {result.expected_change:+.2f}")

            report.append(f"  → {result.interpretation}")

            if result.contraindications:
                report.append("\n  ⚠️ 금기사항:")
                for ci in result.contraindications:
                    report.append(f"    • {ci}")

            if result.warnings:
                report.append("\n  ⚡ 주의사항:")
                for w in result.warnings:
                    report.append(f"    • {w}")

            if result.monitoring_recommendations:
                report.append("\n  📊 모니터링:")
                for m in result.monitoring_recommendations:
                    report.append(f"    • {m}")

            report.append("")

        report.append("=" * 80)
        report.append("※ 본 예측은 문헌 기반 규칙으로 추정한 것이며, 실제 환자 반응은 다를 수 있습니다.")
        report.append("※ 의료진 판단과 함께 사용하시기 바랍니다.")
        report.append("=" * 80)

        return "\n".join(report)




In [ ]:
# ============================================================================
# 사용 예시
# ============================================================================

if __name__ == "__main__":
    # 시뮬레이터 초기화
    simulator = NutritionSimulator()

    # 예시 환자 1: 철결핍성 빈혈
    patient1 = PatientProfile(
        age=72,
        sex='F',
        hemoglobin=9.5,
        ferritin=15.0,
        tsat=12.0,
        ckd_stage=0,
        chronic_inflammation=False
    )

    intervention1 = NutritionIntervention(
        iron_mg=150,
        duration_weeks=4
    )

    print("\n예시 1: 철결핍성 빈혈 환자")
    print("=" * 80)
    results1 = simulator.simulate(patient1, intervention1)
    report1 = simulator.generate_report(patient1, intervention1, results1)
    print(report1)

    # 예시 환자 2: 비타민 D 결핍 + 골절 위험
    patient2 = PatientProfile(
        age=78,
        sex='F',
        vitamin_d=12.0,
        calcium=9.0,
        fracture_risk_high=True,
        ckd_stage=0
    )

    intervention2 = NutritionIntervention(
        vitamin_d_iu=2000,
        calcium_mg=1000,
        duration_weeks=12
    )

    print("\n\n예시 2: 비타민 D 결핍 + 골절 고위험 환자")
    print("=" * 80)
    results2 = simulator.simulate(patient2, intervention2)
    report2 = simulator.generate_report(patient2, intervention2, results2)
    print(report2)

    # 예시 환자 3: CKD + 빈혈 + 저알부민혈증 (복합)
    patient3 = PatientProfile(
        age=75,
        sex='M',
        hemoglobin=10.2,
        ferritin=80.0,
        albumin=2.9,
        ckd_stage=3,
        crp=8.5,
        chronic_inflammation=True
    )

    intervention3 = NutritionIntervention(
        iron_mg=100,
        protein_g=55,
        duration_weeks=8
    )

    print("\n\n예시 3: CKD stage 3 + 빈혈 + 저알부민혈증 환자")
    print("=" * 80)
    results3 = simulator.simulate(patient3, intervention3)
    report3 = simulator.generate_report(patient3, intervention3, results3)
    print(report3)

    # 예시 환자 4: 비타민 C 고용량 (남성 신결석 주의)
    patient4 = PatientProfile(
        age=68,
        sex='M',
        kidney_stone_history=True,
        smoker=True
    )

    intervention4 = NutritionIntervention(
        vitamin_c_mg=1500,
        duration_weeks=12
    )

    print("\n\n예시 4: 흡연 남성 + 신결석 병력 (비타민 C 고용량)")
    print("=" * 80)
    results4 = simulator.simulate(patient4, intervention4)
    report4 = simulator.generate_report(patient4, intervention4, results4)
    print(report4)


예시 1: 철결핍성 빈혈 환자
영양 중재 효과 예측 시뮬레이션 (문헌 기반)

환자: 72세, F

중재 계획:
  - 철분: 150 mg/day
  - 기간: 4주

예측 결과:
--------------------------------------------------------------------------------

【Hemoglobin】
  현재: 9.50
  예상: 10.50 (+1.00)
  → 4주 후 Hgb +1.0 g/dL 변화 예상 (문헌 기반 추정)

  📊 모니터링:
    • Hgb every 2-4 weeks
    • Ferritin/TSAT every 2-3 months

※ 본 예측은 문헌 기반 규칙으로 추정한 것이며, 실제 환자 반응은 다를 수 있습니다.
※ 의료진 판단과 함께 사용하시기 바랍니다.


예시 2: 비타민 D 결핍 + 골절 고위험 환자
영양 중재 효과 예측 시뮬레이션 (문헌 기반)

환자: 78세, F

중재 계획:
  - 비타민 D: 2000 IU/day
  - 칼슘: 1000 mg/day
  - 기간: 12주

예측 결과:
--------------------------------------------------------------------------------

【25(OH)D】
  현재: 12.00
  예상: 42.00 (+30.00)
  → 12주 후 25(OH)D +30.0 ng/mL 증가 예상 (경증 결핍 → 보충 권장)

  📊 모니터링:
    • 25(OH)D and calcium every 3-6 months


【Fracture Risk】
  예상 변화: -15.00
  → 12주 복용 시 골절 위험 약 15% 감소 예상 (Ca+VitD 병용 시) - 고위험군에서 효과 더 뚜렷함

  📊 모니터링:
    • Serum calcium and renal function periodically
    • 변비, 신결석 증상 관찰

※ 본 예측은 문헌 기반 규칙으로 추정한 것이며, 실제 환